# Notebook 03 — Replication of Testori & Lo Iacono (2026)

Replicates Tables 2–3 of the parent paper.

## Modelling choice: logistic regression with session-clustered SEs

The equivalent Stata command from the parent paper's `.do` file is:
```stata
logit extraction_2 ib4.cpr1groupgrp_treat_num age education round, ///
      vce(cluster sessioncode)
lincom 1-3   /* Contrast A: IT&GI vs GI_only */
lincom 2-4   /* Contrast B: IT_only vs control */
```

A GLMM robustness check (random intercepts by session) is run in Notebook 05.

---

**Key gate:** Both contrasts must be positive and significant (p < .05)
before `IT_present` is used as a collapsed indicator in H2–H4.

In [1]:
# ── Cell 1: Load clean data ──────────────────────────────────
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

df_mt = pd.read_csv('data/mturk_clean.csv')
df_pr = pd.read_csv('data/prolific_clean.csv')
datasets = {'mturk': df_mt, 'prolific': df_pr}
PLATFORMS = {'mturk': 'Study 1 (MTurk)', 'prolific': 'Study 2 (Prolific)'}

In [2]:
# ── Cell 2: Replication model — logistic regression with clustered SEs ────
# DV: extraction_2 (binary: 1 = green choice, 0 = non-green).
# Clustered SEs account for non-independence within sessions (level-3),
# matching the intent of: mixed ... || sessioncode: || participantcode:
#
# Fixed effects: three condition dummies (ref = cond_control / NIT&NGI),
#                age, education, round.
# Robustness check with GLMM random intercepts is in Notebook 05.

FORMULA_REP = (
    "extraction_2 ~ "
    "cond_IT_GI + cond_IT_only + cond_GI_only + "
    "age + education + round"
)

replication_models = {}

def fit_replication_model(df, label):
    data = df.dropna(subset=['extraction_2', 'age', 'education', 'round'])

    model = smf.logit(
        FORMULA_REP,
        data=data
    ).fit(
        cov_type='cluster',
        cov_kwds={'groups': data['sessioncode']},
        disp=False
    )

    print(f"\n{'='*55}")
    print(f"  Replication Model — {label}")
    print(f"  N = {len(data):,}  |  Sessions = {data['sessioncode'].nunique()}")
    print(f"{'='*55}")
    print(model.summary().tables[1])

    print("\nOdds Ratios (exp(coef)):")
    print(np.exp(model.params).round(3))

    return model

replication_models['mturk']    = fit_replication_model(df_mt, PLATFORMS['mturk'])
replication_models['prolific'] = fit_replication_model(df_pr, PLATFORMS['prolific'])


  Replication Model — Study 1 (MTurk)
  N = 11,240  |  Sessions = 309
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -1.5668      0.201     -7.812      0.000      -1.960      -1.174
cond_IT_GI       1.0211      0.125      8.196      0.000       0.777       1.265
cond_IT_only     0.6632      0.132      5.007      0.000       0.404       0.923
cond_GI_only     0.7651      0.126      6.070      0.000       0.518       1.012
age              0.0016      0.003      0.498      0.618      -0.005       0.008
education        0.0227      0.031      0.732      0.464      -0.038       0.084
round           -0.0418      0.008     -5.012      0.000      -0.058      -0.025

Odds Ratios (exp(coef)):
Intercept       0.209
cond_IT_GI      2.776
cond_IT_only    1.941
cond_GI_only    2.149
age             1.002
education       1.023
round           0.959
dtype: float64

  Re

In [3]:
# ── Cell 3: contrasts ───────────────────────────
# Contrasts are computed on the log-odds scale (linear combination
# of coefficients), then converted to an odds ratio for reporting.
#
# Contrast A: β(cond_IT_GI) − β(cond_GI_only)
#   "Does IT add value when GI is already present?"
#   Expected: positive → IT increases log-odds of green choice
#             even when green incentive is present.
#
# Contrast B: β(cond_IT_only) vs reference (control = 0)
#   "Does IT add value from a pure baseline?"
#   Expected: positive → IT increases log-odds vs control.
#
# SE for Contrast A: √(SE_IT_GI² + SE_GI_only²)
# (off-diagonal covariance term ignored here; full delta-method
#  version available via model.cov_params() if required)
#
# Both contrasts must be positive and significant (p < .05)
# before IT_present is used as a collapsed indicator in H2–H4.

def compute_contrasts(model, label):
    b  = model.params
    se = model.bse

    contrasts = {
        'Contrast A: β(IT&GI) − β(GI_only)  [Stata: lincom 1-3]': (
            b['cond_IT_GI']  - b['cond_GI_only'],
            np.sqrt(se['cond_IT_GI']**2 + se['cond_GI_only']**2)
        ),
        'Contrast B: β(IT_only) vs. control  [Stata: lincom 2-4]': (
            b['cond_IT_only'],
            se['cond_IT_only']
        ),
    }

    rows = []
    for cname, (beta, stderr) in contrasts.items():
        z    = beta / stderr
        pval = 2 * (1 - stats.norm.cdf(abs(z)))
        OR   = np.exp(beta)
        sig  = ('***' if pval < .001 else '**' if pval < .01
                else '*' if pval < .05 else 'n.s.')
        rows.append({
            'Contrast': cname,
            'log-odds': round(beta, 4),
            'OR':       round(OR, 3),
            'SE':       round(stderr, 4),
            'z':        round(z, 3),
            'p':        round(pval, 4),
            'Sig':      sig
        })

    res = pd.DataFrame(rows).set_index('Contrast')
    print(f"\n{label} — Contrasts (log-odds scale)")
    print(res.to_string())
    return res

c_mt = compute_contrasts(replication_models['mturk'],   PLATFORMS['mturk'])
c_pr = compute_contrasts(replication_models['prolific'], PLATFORMS['prolific'])


Study 1 (MTurk) — Contrasts (log-odds scale)
                                                         log-odds     OR      SE      z       p   Sig
Contrast                                                                                             
Contrast A: β(IT&GI) − β(GI_only)  [Stata: lincom 1-3]     0.2560  1.292  0.1772  1.445  0.1486  n.s.
Contrast B: β(IT_only) vs. control  [Stata: lincom 2-4]    0.6632  1.941  0.1324  5.007  0.0000   ***

Study 2 (Prolific) — Contrasts (log-odds scale)
                                                         log-odds     OR      SE      z       p   Sig
Contrast                                                                                             
Contrast A: β(IT&GI) − β(GI_only)  [Stata: lincom 1-3]     0.2103  1.234  0.2107  0.998  0.3183  n.s.
Contrast B: β(IT_only) vs. control  [Stata: lincom 2-4]    0.5370  1.711  0.1539  3.488  0.0005   ***


In [4]:
# ── Cell 4: Interpretation gate ──────────────────────────────
# Automated pass/fail check before proceeding to moderation models.
# Both contrasts must be (a) positive direction AND (b) p < 0.05.

def gate_check(contrasts, label):
    print(f"\n{label} — Gate check:")
    all_pass = True
    for cname, row in contrasts.iterrows():
        direction = row['log-odds'] > 0
        sig       = row['p'] < 0.05
        status    = "✓ PASS" if (direction and sig) else "✗ FAIL"
        if not (direction and sig):
            all_pass = False
        print(f"  {status}  log-odds={row['log-odds']:+.4f}  "
              f"OR={row['OR']:.3f}  p={row['p']:.4f}")
        print(f"         {cname[:65]}")
    conclusion = 'IS' if all_pass else 'IS NOT'
    print(f"  → IT_present {conclusion} justified as collapsed indicator "
          f"for H2–H4")

gate_check(c_mt, PLATFORMS['mturk'])
gate_check(c_pr, PLATFORMS['prolific'])


Study 1 (MTurk) — Gate check:
  ✗ FAIL  log-odds=+0.2560  OR=1.292  p=0.1486
         Contrast A: β(IT&GI) − β(GI_only)  [Stata: lincom 1-3]
  ✓ PASS  log-odds=+0.6632  OR=1.941  p=0.0000
         Contrast B: β(IT_only) vs. control  [Stata: lincom 2-4]
  → IT_present IS NOT justified as collapsed indicator for H2–H4

Study 2 (Prolific) — Gate check:
  ✗ FAIL  log-odds=+0.2103  OR=1.234  p=0.3183
         Contrast A: β(IT&GI) − β(GI_only)  [Stata: lincom 1-3]
  ✓ PASS  log-odds=+0.5370  OR=1.711  p=0.0005
         Contrast B: β(IT_only) vs. control  [Stata: lincom 2-4]
  → IT_present IS NOT justified as collapsed indicator for H2–H4


In [5]:
# ── Cell 5: Average Marginal Effects (AME) ───────────────────
# Logistic β coefficients are on the log-odds scale and not
# directly comparable across models or platforms.
# Average Marginal Effects (AME) express the effect as a
# change in predicted probability, averaged over all observations.
#
# AME(x_k) = mean over sample of [∂P(Y=1)/∂x_k]
#           = mean of [β_k · p̂_i · (1 − p̂_i)]
# where p̂_i is the predicted probability for observation i.

def compute_ame(model, data, label):
    p_hat  = model.predict(data)              # Pr(Y=1) for each obs
    weight = p_hat * (1 - p_hat)              # logistic derivative weight

    params_of_interest = ['cond_IT_GI', 'cond_IT_only', 'cond_GI_only',
                           'age', 'education', 'round']
    rows = []
    for var in params_of_interest:
        if var in model.params:
            ame = model.params[var] * weight.mean()
            rows.append({'Variable': var, 'AME': round(ame, 4)})

    df_ame = pd.DataFrame(rows)
    print(f"\n{label} — Average Marginal Effects (Δ Pr[green])")
    print(df_ame.to_string(index=False))
    return df_ame

for name, m in replication_models.items():
    data = datasets[name].dropna(
        subset=['extraction_2', 'age', 'education', 'round'])
    compute_ame(m, data, PLATFORMS[name])


Study 1 (MTurk) — Average Marginal Effects (Δ Pr[green])
    Variable     AME
  cond_IT_GI  0.1943
cond_IT_only  0.1262
cond_GI_only  0.1456
         age  0.0003
   education  0.0043
       round -0.0079

Study 2 (Prolific) — Average Marginal Effects (Δ Pr[green])
    Variable     AME
  cond_IT_GI  0.1671
cond_IT_only  0.1117
cond_GI_only  0.1233
         age  0.0007
   education  0.0018
       round -0.0098
